## Draft Prospect Scouting Pipeline (DLT + AI)

This notebook defines a multi-step Delta Live Tables (DLT) pipeline that processes and enriches draft prospect data from FanGraphs.

### 🔄 Pipeline Overview

1. **Bronze Ingestion (Step 1):**  
   Ingests raw JSON data from a cloud volume using AutoLoader, partitioned by season and board type (e.g., `mlb`). Includes metadata capture for auditability.

2. **Silver Enrichment (Steps 2–4):**  
   Cleans and enriches each row with human and AI-generated scouting summaries. The AI summary is generated using the `ai_query()` SQL function and the `databricks-llama-4-maverick` model.

3. **Vector Index Preparation (Steps 5–7):**  
   Creates a slimmed-down, text-focused view of enriched prospects designed for embedding and retrieval in a vector search index (e.g., Databricks Vector Search with Delta Sync).

This notebook assumes Unity Catalog and LLM function support are enabled in your workspace. The output is suitable for downstream tasks like Retrieval-Augmented Generation (RAG) or search-based scouting interfaces.


In [0]:
import dlt  # Import Delta Live Tables for data pipeline management
import pyspark.sql.functions as F  # Import PySpark SQL functions for DataFrame operations

In [0]:
# Step 1: Bronze
@dlt.table(
    name="bronze_draft_prospects",
    comment="Raw prospect data from FanGraphs API (bronze layer)",
    table_properties={
        "quality": "bronze"
    }
)
def bronze_draft_prospects():
    season = spark.conf.get("season", "2025")
    board_type = spark.conf.get("board_type", "mlb")
    path = f"/Volumes/alexander_booth/rag_demo/prospect_data/source=fangraphs/board_type={board_type}/season={season}"
    
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("multiLine", "true")
            .load(path)
            .withColumn("load_date", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path"))
            .withColumn("file_mod_time", F.col("_metadata.file_modification_time"))
            .withColumn("file_size", F.col("_metadata.file_size"))
    )

### Enrich Draft Prospects with AI-Generated Scouting Reports

This step reads streaming prospect data from the bronze layer and generates enriched scouting reports.

- If a human-written summary (`Summary`) exists, it's included as a `"Full Report"`.
- If a summary is present, an AI-generated scouting report is also created using the `ai_query()` SQL function.
  - The entire row is serialized as a JSON blob and passed to the `databricks-llama-4-maverick` model.
  - The prompt instructs the model to write a full scouting report in paragraphs using the data in a given row
- The outputs are combined into a `combined_text` field used downstream.
- Rows missing an `ID` are dropped with the `@dlt.expect_or_drop` constraint.


In [0]:
# Step 2: Enriched view
@dlt.view
@dlt.expect_or_drop("valid_ID", "ID IS NOT NULL")
def enriched_draft_prospects():
    raw = dlt.read_stream("bronze_draft_prospects")

    # Text Blocks
    summary_block = F.when(
        F.col("Summary").isNotNull(), F.concat(F.lit("Full Report:\n"), F.col("Summary"))
    )

    ai_summary_block = F.when(
        F.col("Summary").isNotNull(),
        F.concat(F.lit("AI Summary:\n"),
        F.expr(
            """
            ai_query(
                'databricks-llama-4-maverick',
                'You are a baseball scouting assistant. Given this json blob, create a full scouting report using complete sentences and paragraphs. Note that Athleticisim is graded -2 to 2, with "null" being average, -1 being below average, etc. ' || to_json(struct(*))
            )
            """
        ))
    )

    return (
    raw.withColumn("primary_key", F.concat_ws("_", F.col("ID"), F.col("Season")))
       .withColumn("combined_text", F.concat_ws("\n\n", summary_block, ai_summary_block))
       .drop(
           "_rescued_data",
           "file_mod_time",
           "file_size"
       )
    )


# Step 3: Register the target table
dlt.create_target_table(
    name="silver_draft_prospects",
    comment="Table storing silver layer scouting reports for prospects",
    table_properties={
        "delta.enableChangeDataFeed": "true",
        "quality": "silver"
    }
)

# Step 4: Apply merge logic to that table
dlt.apply_changes(
    target="silver_draft_prospects",
    source="enriched_draft_prospects",
    keys=["primary_key"],
    sequence_by=F.col("ingest_date"),
    stored_as_scd_type=1
)

### Prepare Slimmed Prospect View for Vector Indexing

These steps generate and maintain a lightweight, searchable subset of enriched prospect data designed specifically for embedding and vector search.

- **Step 5** creates a view (`silver_draft_prospects_index_input_changes`) that selects only the necessary fields from the `silver_draft_prospects` table — including `combined_text`, the AI-enriched scouting report.
- **Step 6** registers a physical Delta table (`silver_draft_prospects_index_input`) that serves as the source for downstream embedding and vector indexing (e.g., via Databricks Vector Search + Delta Sync).
- **Step 7** uses `dlt.apply_changes` to apply change data capture (CDC) logic:
  - Tracks updates based on the `primary_key` and `ingest_date`.
  - Maintains a Type 1 Slowly Changing Dimension (overwrite on change).
  - Ensures the index input stays current with the latest scouting report updates.


In [0]:
# Step 5: Narrow view from silver_prospects for index
@dlt.table
def silver_draft_prospects_index_input_changes():
    return dlt.read("silver_draft_prospects").select(
        "primary_key", "Season", "ID", "minorMasterId", "playerName", "combined_text", "ingest_date"
    )

# Step 6: Register physical Delta table for vector index input
dlt.create_target_table(
    name="silver_draft_prospects_index_input",
    comment="Slimmed down version of silver_draft_prospects used for embedding and vector indexing",
    table_properties={
        "delta.enableChangeDataFeed": "true",
        "quality": "silver"
    }
)

# Step 7: Apply merge logic to populate it
dlt.apply_changes(
    target="silver_draft_prospects_index_input",
    source="silver_draft_prospects_index_input_changes",
    keys=["primary_key"],
    sequence_by=F.col("ingest_date"),
    stored_as_scd_type=1
)